# Hybrid 10 · The project scripts and the artefacts they produced

**Objective:** explain what each script in `ml/grading/scripts/` does, where it sits in the pipeline, and check that its saved outputs are consistent with the data used in the notebooks.

**Read-only:** no script is re-run here, because every script except `modeling_setup.py` **writes or overwrites** files (image folders, audit tables, or the deployed model artefacts in `ml-service/models/`). This notebook only reads their outputs. **Runtime:** a few seconds.

| Order | Script | What it does | Main outputs | Covered in |
|---|---|---|---|---|
| 1 | `extract_sp_images.py` | Copies each tissue's specular (`_SP.jpg`) image from the raw image folder by matching tissue IDs | `data/images_sp_v2/` + `image_manifest.csv`, missing/multiple/unmatched lists | §1 |
| 2 | `audit_sp_images.py` | Hashes every copied image; finds byte-identical duplicates, tissues without images, images without records, grade conflicts | `data/image_audit_v2/*.csv` | §1 |
| 3 | `prepare_image_dataset.py` | Keeps one image per tissue (collapsing identical copies, flagging differing ones for review) into the clean image set | `image/data/processed/clean_image_manifest.csv`, `images/`, `duplicate_review.csv` | §1, Images 01 |
| 4 | `modeling_setup.py` | Shared module (writes nothing once its files exist): feature lists, preprocessing, holdout split, donor inference, donor-grouped folds | (`holdout_split.csv`, `donor_grouped_folds.csv` on first use only) | Tabular 03, Hybrid 01–02 |
| 5 | `export_feature_schema.py` | Exports the 24 features, their types, allowed categories and training ranges for input validation in `ml-service` | `ml-service/models/grading_feature_schema.json` | §2 |
| 6 | `train_grading_model.py` | Refits the depth-2 ECD tree selected by notebook 13 and exports it as cut points + band counts (no tissue-level data) | `ml-service/models/grading_ecd_tree.json`, golden test file | §3 |
| 7 | `train_hybrid_grading_model.py` | Re-tunes and fits the deployed hybrid RF on all 551 K3 tissues (Amendment 2) | `ml-service/models/grading_hybrid_rf.{joblib,json}`, golden test file | Hybrid 09 (re-run in memory, identical) |
| 8 | `train_domain_validator.py` | Trains the "is this a corneal specular image?" gate that runs before any grading | `ml-service/models/domain_validator/` | §4 |

Set up paths.

In [1]:
import hashlib
import json
import sys
from pathlib import Path

GRADING_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "modeling_setup.py").is_file())
sys.path[:0] = [str(GRADING_DIR / "scripts"), str(GRADING_DIR / "notebooks_cleaned")]

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

import cleaned_setup as cs
import modeling_setup as ms

MODELS = cs.DEPLOYED_MODEL_DIR
EXTRACTION_DIR = GRADING_DIR / "data" / "images_sp_v2"
AUDIT_DIR = GRADING_DIR / "data" / "image_audit_v2"

## 1. Image extraction, audit and preparation (scripts 1–3)

Follow the image counts through the three scripts: copied files, tissues without an image, tissues with several source files, and the final one-image-per-tissue manifest.

In [2]:
extraction = pd.read_csv(EXTRACTION_DIR / "image_manifest.csv")
summary = pd.read_csv(AUDIT_DIR / "tissue_image_summary.csv")
clean = pd.read_csv(cs.MANIFEST_PATH)
duplicates = pd.read_csv(AUDIT_DIR / "exact_duplicate_files.csv")
pd.Series({
    "Tissue records in the cleaned dataset": len(summary),
    "Image files copied (script 1)": len(extraction),
    "Tissues with at least one image": extraction["Normalized_ID"].nunique(),
    "Tissues without an image (script 1)": len(pd.read_csv(EXTRACTION_DIR / "missing_images.csv")),
    "Tissues matched to several source files (script 1)": len(pd.read_csv(EXTRACTION_DIR / "multiple_matches.csv")),
    "Images without a tissue record (script 2)": len(pd.read_csv(AUDIT_DIR / "images_without_dataset_record.csv")),
    "Tissues whose copies differ in content (script 2)": int((summary["Distinct_Image_Hashes"] > 1).sum()),
    "Tissues in the clean manifest (script 3)": len(clean),
}).to_frame("Count")

,Count
Tissue records in the cleaned dataset,1054
Image files copied (script 1),1524
Tissues with at least one image,1026
Tissues without an image (script 1),28
Tissues matched to several source files (script 1),494
Images without a tissue record (script 2),0
Tissues whose copies differ in content (script 2),2
Tissues in the clean manifest (script 3),1024


Consistency checks across the three scripts: the tissues whose copies differ are exactly the two flagged for manual review, and script 3 left them out, so the clean manifest is every tissue with an image minus those two (1,026 − 2 = 1,024).

In [3]:
differing = set(summary.loc[summary["Distinct_Image_Hashes"] > 1, "Normalized_ID"])
print("Tissues with differing copies = flagged review IDs:", differing == {cs.normalize_id(i) for i in cs.REVIEW_IDS})
print("Clean manifest = tissues with images minus the 2 flagged:", set(extraction["Normalized_ID"]) - differing == set(clean["Normalized_ID"]))
print("Flagged tissues present in the clean manifest:", len(differing & set(clean["Normalized_ID"])))
print("(Image–record grade agreement is checked in Images 01.)")

Tissues with differing copies = flagged review IDs: True
Clean manifest = tissues with images minus the 2 flagged: True
Flagged tissues present in the clean manifest: 0
(Image–record grade agreement is checked in Images 01.)


Scripts 1–3 contain absolute Windows paths, including the raw image folder outside the repository (`…/research/CellChechD`). They cannot be re-run on another machine without editing those paths. Their outputs above (git-ignored) are what the notebooks use. Images 01 verifies every clean image against its recorded SHA-256.

In [4]:
for script in ["extract_sp_images.py", "audit_sp_images.py", "prepare_image_dataset.py"]:
    text = (GRADING_DIR / "scripts" / script).read_text(encoding="utf-8")
    absolute = [line.strip() for line in text.splitlines() if 'r"C:' in line]
    print(f"{script}: {len(absolute)} hard-coded absolute path(s)")

extract_sp_images.py: 3 hard-coded absolute path(s)
audit_sp_images.py: 1 hard-coded absolute path(s)
prepare_image_dataset.py: 1 hard-coded absolute path(s)


## 2. Feature schema used to validate application inputs (script 5)

The schema must list the same 24 features, in the same order, as `modeling_setup.ALL_FEATURES`. Its allowed categories and ranges must match the labelled training data as the models read it.

In [5]:
schema = json.loads((MODELS / "grading_feature_schema.json").read_text(encoding="utf-8"))
labelled = ms.load_labelled_data()
by_name = {f["name"]: f for f in schema["features"]}
print("Same features in the same order:", schema["order"] == ms.ALL_FEATURES == [f["name"] for f in schema["features"]])
rows = []
for name in ms.ALL_FEATURES:
    f = by_name[name]
    if f["type"] == "category":
        ok = sorted(f["allowed"]) == sorted(labelled[name].dropna().astype(str).unique())
        detail = ", ".join(f["allowed"])
    elif f.get("training_range"):
        ok = f["training_range"] == {"min": float(labelled[name].min()), "max": float(labelled[name].max())}
        detail = f"{f['training_range']['min']:g}–{f['training_range']['max']:g}"
    else:
        ok, detail = True, "0 / 1"
    rows.append({"Feature": name, "Type": f["type"], "Allowed / training range": detail, "Matches training data": ok})
table = pd.DataFrame(rows)
print("All features match the training data:", table["Matches training data"].all())
table

Same features in the same order: True
All features match the training data: True


,Feature,Type,Allowed / training range,Matches training data
0,ECD_CD_cells_per_mm2,number,676–4484,True
1,CV,number,24–74,True
2,HEX_percent,number,27–76,True
3,SD,number,86–753,True
4,Total_Cell_Count,number,10–204,True
5,Total_Area_um2,number,5071–100692,True
6,Stroma_Is_Clear,binary,0 / 1,True
7,Stroma_Is_Compact,binary,0 / 1,True
8,Stroma_Has_Arcus,binary,0 / 1,True
9,Stroma_Has_Edema,binary,0 / 1,True


Two points to know about the schema:

- Categories come from the data **as loaded**, so "None" is not an allowed category: "recorded absent" is sent as missing, exactly as the models were trained (Tabular 01, §6).
- Its descriptive field `deployed_model_uses` still says only ECD. It was written when the ECD tree was served, before Amendment 2. The deployed hybrid uses all 24 features plus the image (Hybrid 09).

In [6]:
print("deployed_model_uses:", schema["deployed_model_uses"], "| training rows:", schema["training_rows"])
print("Cross-field check:", schema["cross_field_checks"][0]["rule"], "->", schema["cross_field_checks"][0]["on_failure"])

deployed_model_uses: ['ECD_CD_cells_per_mm2'] | training rows: 1036
Cross-field check: ECD_CD_cells_per_mm2 ≈ Total_Cell_Count / Total_Area_um2 × 10^6 (within 1%) -> warning (values reviewed, not rejected)


## 3. The exported ECD reference tree (script 6)

Refit the depth-2 tree on the 470 K4 tissues (as the script does) and compare its cut points and per-band grade counts with the exported spec that `ml-service` shows beside every grade.

In [7]:
k3, _, has_ecd = cs.primary_cohort()
ecd = k3.loc[has_ecd, cs.ECD].to_numpy().reshape(-1, 1)
grades = k3.loc[has_ecd, ms.TARGET].to_numpy()
tree = DecisionTreeClassifier(**cs.TREE_PARAMS).fit(ecd, grades)
cuts = sorted(float(t) for t in tree.tree_.threshold[tree.tree_.feature >= 0])
spec = json.loads((MODELS / "grading_ecd_tree.json").read_text(encoding="utf-8"))
band = np.digitize(ecd.ravel(), cuts, right=True)  # band i: cut[i-1] < ECD <= cut[i]
counts = [{g: int(((band == i) & (grades == g)).sum()) for g in cs.CLASSES} for i in range(3)]
print("Cut points refit / exported:", cuts, spec["cut_points"], "| equal:", cuts == spec["cut_points"])
print("Band grade counts equal:", counts == [b["training_counts"] for b in spec["bands"]])
print("Spec trained on dataset SHA-256:", spec["training"]["data_sha256"] == hashlib.sha256(ms.DATA_PATH.read_bytes()).hexdigest())

Cut points refit / exported: [2232.0, 2497.0] [2232.0, 2497.0] | equal: True
Band grade counts equal: True
Spec trained on dataset SHA-256: True


## 4. The corneal-image domain validator (script 8)

Before any grade is produced, the application checks that the upload is a specular microscopy image of the corneal endothelium:
- frozen ImageNet ResNet-18 features (ONNX) on a greyscale image;
- standardisation + logistic regression;
- thresholds set on a validation split only.

It does **not** grade and does **not** assess image quality.

In [8]:
validator = json.loads((MODELS / "domain_validator" / "classifier.json").read_text(encoding="utf-8"))
evaluation = json.loads((MODELS / "domain_validator" / "evaluation.json").read_text(encoding="utf-8"))
print("Question:", validator["question"])
print("Not assessed:", validator["not_assessed"])
print("Classifier:", validator["training"]["classifier"], "| thresholds:", {k: round(v, 3) for k, v in validator["thresholds"].items() if isinstance(v, float)})
pd.DataFrame(validator["training"]["counts"]).T

Question: Is this a specular microscopy image of the corneal endothelium?
Not assessed: ['image quality', 'cell visibility / cell detail', 'disease']
Classifier: StandardScaler + LogisticRegression(C=1.0, class_weight='balanced') | thresholds: {'reject_below': 0.5, 'accept_at_or_above': 0.61, 'max_validation_negative_score': 0.61}


,coco,specular_microscopy,synthetic_degenerate,synthetic_document,synthetic_screenshot
train,2990,619,420,238,242
validation,997,208,140,86,74
test,995,197,140,76,84


Held-out test results, reported once. Non-corneal images by type, specular images by grade, and deliberately degraded specular images:

In [9]:
rows = [{"Test images": f"Non-corneal: {k}", "n": v["n"], "Accepted as corneal": v["accepted_count"]}
        for k, v in evaluation["negatives"].items() if ":" not in k]
rows += [{"Test images": f"Specular: {k}", "n": v["n"], "Accepted as corneal": v["accepted_count"]} for k, v in evaluation["positives"].items()]
rows += [{"Test images": f"Specular, degraded: {k}", "n": v["n"], "Accepted as corneal": v["accepted_count"]} for k, v in evaluation["positives_degraded"].items()]
pd.DataFrame(rows)

,Test images,n,Accepted as corneal
0,Non-corneal: all,1295,0
1,Non-corneal: coco,995,0
2,Non-corneal: synthetic_document,76,0
3,Non-corneal: synthetic_screenshot,84,0
4,Non-corneal: synthetic_degenerate,140,0
5,Specular: all,197,197
6,Specular: grade A+,111,111
7,Specular: grade A,52,52
8,Specular: grade B,30,30
9,"Specular, degraded: gaussian_blur_r2",197,197


- **0 of 1,295** non-corneal test images were accepted, and **197 of 197** specular images were accepted, across all three grades.
- Heavily blurred and downscaled specular images were also accepted. This confirms that the validator checks the *kind* of image, not its quality.
- **History:** version 1 accepted blank grey, black and noise images; a test written before training caught it, and version 2 added such negatives (`evaluation_v1.json`).
- **Limitation:** its positive images come from `clean_image_manifest.csv`, which contains the one blank white image found in Images 01.
- This is an engineering evaluation only: other microscopes, near-domain images and image quality were not tested.

## Conclusion

- Scripts 1–3 built the image set (1,024 tissues with one verified image each). `modeling_setup.py` is the shared definition of features, preprocessing and folds. Scripts 5–8 export what the application runs: the input schema, the ECD reference tree, the hybrid model and the image gate.
- Their saved outputs are consistent with the data used in the notebooks. The ECD tree spec equals a fresh refit, the schema matches the 24 training features, and the hybrid file reproduces exactly (Hybrid 09).
- **Do not run scripts 1–3 or 5–8 casually:** they overwrite git-ignored data or the deployed artefacts in `ml-service/models/`.
- **Open issues:** scripts 1–3 use absolute paths; the schema's `deployed_model_uses` text is outdated.